# Board Diagnostic — Self Contained
Run cells 1→2→3 in order.

## Cell 1 — Config

In [1]:
RP_IP    = "192.168.0.99"
SSH_USER = "root"
SSH_PASS = "root"
print("OK")

OK


## Cell 2 — Kill + restart board cleanly

In [2]:
import paramiko, time

def ssh(cmd):
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS, timeout=10)
    _, o, e = c.exec_command(cmd)
    out = o.read().decode().strip()
    c.close()
    return out

ssh("pkill -f RunLock.py")
time.sleep(2)
ssh("PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py > /tmp/runlock.log 2>&1 &")
time.sleep(5)

ports = ssh("ss -tlnp | grep -E '5000|5065|5066'")
log   = ssh("cat /tmp/runlock.log")
print("Ports:", ports or "NONE")
print("Log:", log or "(empty)")

Ports: LISTEN 0      128     192.168.0.99:5000      0.0.0.0:*    users:(("python3",pid=5228,fd=7))
Log: (empty)


In [3]:
# Paste output here
with open("communication.py") as f:
    src = f.read()

# Check exactly what's in the while loop
idx = src.find("if loop_action:")
print("=== loop_action block ===")
print(src[idx:idx+800])
print()
print("connect_socket method:")
idx2 = src.find("def connect_socket")
print(src[idx2:idx2+300])

=== loop_action block ===
if loop_action:
                self.loop_running = True

            while True:
                sleep(0)
                # After port 5000 response arrives (message.selkey set),
                # connect to port 5065. action_start_scan now returns "started"
                # immediately (non-blocking), so message.selkey is set fast.
                if loop_action and self.loop_running and self.lsock is None:
                    if message.selkey is not None:
                        # Response received â€” board has started loop thread.
                        # Retry port 5065 for up to 15s.
                        laddr = (self.addr[0], 5065)
                        for _retry in range(15):
                            sleep(1)
                            self.lsock = self.connect_socket

connect_socket method:
def connect_socket(self, addr):
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)  # IPv4 TCP socket
        sock.settimeout(5)  # 5 s

## Cell 3 — Send start_scan and watch board log every second

In [ ]:
import paramiko, time, socket, struct, json

def ssh(cmd):
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS, timeout=10)
    _, o, _ = c.exec_command(cmd)
    out = o.read().decode().strip()
    c.close()
    return out

# Build raw start_scan message
content = json.dumps({"action":"start_scan","value":{"amplitude":0.7,"offset":0.0}},
                     ensure_ascii=False).encode()
jh = json.dumps({"byteorder":"little","content-type":"text/json",
                  "content-encoding":"utf-8","content-length":len(content)},
                ensure_ascii=False).encode()
raw = struct.pack(">H", len(jh)) + jh + content

# Snapshot log before
prev = ssh("cat /tmp/runlock.log")

# Connect + send
print("Connecting to port 5000...")
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(5)
sock.connect((RP_IP, 5000))
sock.sendall(raw)
sock.setblocking(False)
print(f"Sent {len(raw)} bytes\n")
print("Board log output (line by line):")
print("-"*50)

for i in range(25):
    time.sleep(1)
    cur   = ssh("cat /tmp/runlock.log")
    ports = ssh("ss -tlnp | grep -E '5065|5066'")
    new   = cur[len(prev):].strip()
    if new:
        for line in new.splitlines():
            print(f"  t={i+1}s: {line}")
        prev = cur
    else:
        print(f"  t={i+1}s: (no output)")
    if ports:
        print(f"\n  *** PORTS OPEN: {ports} ***")
        break

print("-"*50)
print("Final log:")
print(ssh("cat /tmp/runlock.log"))
sock.close()